[📚 Spis lekcji](https://github.com/TWOJ_LOGIN/NAZWA_REPO#spis-lekcji) &nbsp;·&nbsp; **Lekcja 1 z 2** &nbsp;·&nbsp; [Lekcja 2: Prędkość →](https://colab.research.google.com/github/TWOJ_LOGIN/NAZWA_REPO/blob/main/Kinematyka_2_predkosc.ipynb)

# 🐝 Kinematyka – część 1: Gdzie jest pszczoła?

W tym zeszycie nauczysz się **opisywać położenie ciała** w przestrzeni – i przy okazji zrobisz pierwsze kroki w Pythonie.

**Jak z niego korzystać?** Komórki z kodem uruchamiasz przyciskiem ▶ po lewej stronie (albo `Shift + Enter`). Rób to po kolei, od góry do dołu. Nic się nie zepsuje – możesz śmiało zmieniać liczby i patrzeć, co się stanie.

> 💾 Chcesz zachować swoje zmiany? Wybierz *Plik → Zapisz kopię na Dysku*.

---
## 1. Układ odniesienia

Powiedzieć „pszczoła jest tutaj" to za mało. **Gdzie jest „tutaj"?** Żeby opisać położenie, musimy się umówić:

1. **Od czego mierzymy** – wybieramy punkt, który nazywamy *początkiem układu współrzędnych*.
2. **W jakich kierunkach mierzymy** – wybieramy trzy prostopadłe osie: $x$, $y$ i $z$.

Taki zestaw nazywamy **układem odniesienia** (tu: układem kartezjańskim). Położenie pszczoły to trzy liczby:

- $x$ – ile metrów od początku w kierunku osi $x$,
- $y$ – ile metrów w kierunku osi $y$,
- $z$ – ile metrów w górę, w kierunku osi $z$.

Liczby mogą być ujemne – wtedy jesteśmy „po drugiej stronie" początku układu.

## 2. Przygotowanie

Poniższa komórka przygotowuje rysowanie pszczoły. Na razie tylko ją uruchom – to kod pomocniczy, do którego wrócimy pod koniec zeszytu. 🙂

In [ ]:
#@title 🔧 Kod pomocniczy – rysowanie (uruchom i nie przejmuj się nim, wrócimy do niego później)
import numpy as np
import plotly.graph_objects as go

R_PSZCZOLY = 0.25   # promień pszczoły [m]

# kamera: widok domyślny i widoki z różnych stron
WIDOKI = {
    "↺ Reset widoku": dict(eye=dict(x=1.6, y=-1.6, z=1.0), up=dict(x=0, y=0, z=1)),
    "Z góry":        dict(eye=dict(x=0.0, y=0.001, z=2.6), up=dict(x=0, y=1, z=0)),
    "Z przodu":      dict(eye=dict(x=0.0, y=-2.6, z=0.0), up=dict(x=0, y=0, z=1)),
    "Z boku":        dict(eye=dict(x=2.6, y=0.0, z=0.0), up=dict(x=0, y=0, z=1)),
}

def pszczola(p, R=R_PSZCZOLY):
    """Kula w żółto-czarne paski ze środkiem w punkcie p."""
    x, y, z = p
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(0, np.pi, 40))
    return go.Surface(
        x=x + R*np.cos(u)*np.sin(v), y=y + R*np.sin(u)*np.sin(v), z=z + R*np.cos(v),
        surfacecolor=(np.floor(v/(np.pi/7)) % 2),
        colorscale=[[0, "#FFD400"], [0.5, "#FFD400"], [0.5, "#1a1a1a"], [1, "#1a1a1a"]],
        cmin=0, cmax=1, showscale=False, hoverinfo="skip",
        lighting=dict(ambient=0.9, diffuse=0.3, specular=0.05),
    )

def strzalka(poczatek, w, kolor, dl_grotu=0.6, r_grotu=0.18):
    """Strzałka-wektor: trzon (linia) + grot (stożek z trójkątów). Czubek leży w punkcie poczatek + w."""
    poczatek = np.asarray(poczatek, float)
    w = np.asarray(w, float)
    d = np.linalg.norm(w)
    e = w / d if d > 1e-9 else np.array([0, 0, 1.0])
    dl = min(dl_grotu, d)                          # krótkie wektory dostają mniejszy grot
    r = r_grotu * dl / dl_grotu
    ostrze = poczatek + w
    podstawa = ostrze - e * dl
    trzon = go.Scatter3d(x=[poczatek[0], podstawa[0]], y=[poczatek[1], podstawa[1]],
                         z=[poczatek[2], podstawa[2]], mode="lines",
                         line=dict(color=kolor, width=7), hoverinfo="skip")
    a = np.cross(e, [1, 0, 0]) if abs(e[0]) < 0.9 else np.cross(e, [0, 1, 0])
    a = a / np.linalg.norm(a)
    b = np.cross(e, a)
    t = np.linspace(0, 2*np.pi, 16, endpoint=False)
    kolo = podstawa + r*(np.outer(np.cos(t), a) + np.outer(np.sin(t), b))
    punkty = np.vstack([ostrze, kolo, podstawa])   # 0 = czubek, 1..16 = okrąg, 17 = środek podstawy
    n = len(t)
    boki = [(0, 1 + k, 1 + (k + 1) % n) for k in range(n)]
    spod = [(n + 1, 1 + (k + 1) % n, 1 + k) for k in range(n)]
    ij = np.array(boki + spod)
    grot = go.Mesh3d(x=punkty[:, 0], y=punkty[:, 1], z=punkty[:, 2],
                     i=ij[:, 0], j=ij[:, 1], k=ij[:, 2], color=kolor, flatshading=True,
                     lighting=dict(ambient=0.8, diffuse=0.4), hoverinfo="skip")
    return [trzon, grot]

def osie_ukladu(L):
    return go.Scatter3d(x=[-L, L, None, 0, 0, None, 0, 0], y=[0, 0, None, -L, L, None, 0, 0],
                        z=[0, 0, None, 0, 0, None, -L, L], mode="lines",
                        line=dict(color="gray", width=3), hoverinfo="skip")

def menu_widokow():
    return dict(type="buttons", direction="right", x=0.0, xanchor="left", y=1.0, yanchor="bottom",
                buttons=[dict(label=n, method="relayout", args=[{"scene.camera": k}])
                         for n, k in WIDOKI.items()])

def menu_ukrywania(opcje, ile, adnotacje=None):
    """Lista rozwijana 'co pokazać'. opcje = [(napis, [indeksy do ukrycia], czy_ukryc_etykiete), ...]"""
    przyciski = []
    for napis, ukryte, ukryj_etykiete in opcje:
        widoczne = [i not in ukryte for i in range(ile)]
        zmiana_ukladu = {} if adnotacje is None else {"scene.annotations": [] if ukryj_etykiete else adnotacje}
        przyciski.append(dict(label=napis, method="update", args=[{"visible": widoczne}, zmiana_ukladu]))
    return dict(type="dropdown", direction="down", x=1.0, xanchor="right", y=1.0, yanchor="bottom",
                showactive=True, buttons=przyciski)

def zrob_uklad(tytul, zakres, dtick, adnotacje=(), menu=(), sliders=(), gora=70, dol=0):
    def os_(nazwa):
        return dict(title=nazwa, range=[-zakres, zakres], dtick=dtick)
    return go.Layout(
        title=dict(text=tytul, x=0.5, y=0.98, yanchor="top"),
        scene=dict(xaxis=os_("x [m]"), yaxis=os_("y [m]"), zaxis=os_("z [m]"), aspectmode="cube",
                   camera=WIDOKI["↺ Reset widoku"], annotations=list(adnotacje)),
        height=680, margin=dict(l=0, r=0, t=gora, b=dol), showlegend=False,
        updatemenus=[menu_widokow()] + list(menu), sliders=list(sliders),
        uirevision="stala",   # zachowuje obrót wykresu podczas animacji i zmian
    )

L = 5   # osie rysujemy od -L do L

def zbuduj_slady(x, y, z):
    """0 = pszczoła, 1-2 = wektor r (trzon, grot), 3 = składowe, 4 = osie."""
    slady = [pszczola((x, y, z))]
    slady += strzalka([0, 0, 0], [x, y, z], "red")          # grot kończy się w środku pszczoły
    slady.append(go.Scatter3d(x=[0, x, x, x], y=[0, 0, y, y], z=[0, 0, 0, z], mode="lines",
                              line=dict(color="steelblue", width=4, dash="dash"), hoverinfo="skip"))
    slady.append(osie_ukladu(L))
    return slady

def tytul(x, y, z):
    return f"<b>r</b> = ({x:g}; {y:g}; {z:g}) m"

def etykieta_r(x, y, z):
    return [dict(x=x/2, y=y/2, z=z/2 + 0.4, text="<b>r</b>", showarrow=False,
                 font=dict(color="red", size=24))]

def uklad(x, y, z, z_lista=True):
    menu = []
    if z_lista:
        menu.append(menu_ukrywania(
            [("Pokaż wszystko", [], False), ("Ukryj pszczołę", [0], False),
             ("Ukryj wektor r", [1, 2], True), ("Ukryj składowe", [3], False)],
            ile=5, adnotacje=etykieta_r(x, y, z)))
    return zrob_uklad(tytul(x, y, z), L, 1, adnotacje=etykieta_r(x, y, z), menu=menu)

def rysuj_pszczole(x, y, z):
    go.Figure(data=zbuduj_slady(x, y, z), layout=uklad(x, y, z)).show()

def pszczola_z_suwakami():
    from ipywidgets import FloatSlider, Checkbox, VBox, HBox
    from IPython.display import display
    try:                                   # nowe wersje plotly potrzebują pakietu anywidget
        import anywidget
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anywidget"])
    try:                                   # w Colabie wykres-widżet wymaga tego włączenia
        from google.colab import output
        output.enable_custom_widget_manager()
    except Exception:
        pass
    sx = FloatSlider(min=-5, max=5, step=0.5, value=3, description="x [m]")
    sy = FloatSlider(min=-5, max=5, step=0.5, value=2, description="y [m]")
    sz = FloatSlider(min=-5, max=5, step=0.5, value=1, description="z [m]")
    pokaz_pszczole = Checkbox(value=True, description="Pokaż pszczołę")
    pokaz_wektor = Checkbox(value=True, description="Pokaż wektor r")
    wykres = go.FigureWidget(data=zbuduj_slady(3, 2, 1), layout=uklad(3, 2, 1, z_lista=False))

    def aktualizuj(zmiana=None):
        x, y, z = sx.value, sy.value, sz.value
        nowe = zbuduj_slady(x, y, z)
        with wykres.batch_update():
            for stary, nowy in zip(wykres.data, nowe):
                dane = nowy.to_plotly_json()
                dane.pop("type", None)
                stary.update(dane)
            wykres.data[0].visible = pokaz_pszczole.value
            wykres.data[1].visible = pokaz_wektor.value
            wykres.data[2].visible = pokaz_wektor.value
            wykres.layout.title.text = tytul(x, y, z)
            wykres.layout.scene.annotations = etykieta_r(x, y, z) if pokaz_wektor.value else []

    for w in (sx, sy, sz, pokaz_pszczole, pokaz_wektor):
        w.observe(aktualizuj, names="value")
    display(VBox([sx, sy, sz, HBox([pokaz_pszczole, pokaz_wektor]), wykres]))

## 3. Pierwsza pszczoła

W Pythonie możemy nadać liczbie **nazwę**. Robimy to znakiem `=`. Poniżej mówimy: *pszczoła jest 3 m od początku w kierunku $x$, 2 m w kierunku $y$ i 1 m nad podłogą*.

Uruchom komórkę:

In [ ]:
x = 3
y = 2
z = 1

rysuj_pszczole(x, y, z)

### 🖱️ Jak obsługiwać wykres?
- **Obrót:** przeciągnij wykres myszą (na telefonie – palcem).
- **Przybliżanie:** kółko myszy (na telefonie – dwa palce).
- **Przesuwanie:** przytrzymaj `Shift` i przeciągaj.
- **Gotowe widoki:** przyciski *Z góry*, *Z przodu*, *Z boku*. Przycisk **↺ Reset widoku** wraca do widoku początkowego.
- **Co pokazać:** lista rozwijana w prawym górnym rogu pozwala **ukryć pszczołę, wektor $\vec{r}$ albo linię składowych**, żeby przyjrzeć się wybranemu elementowi.

*(Na wykresie wektor oznaczamy pogrubioną literą **r**, bo wykresy 3D nie rysują strzałki nad literą. W tekście zeszytu nadal piszemy $\vec{r}$.)*

Na rysunku widzisz:
- **żółto-czarną kulę** – pszczołę (traktujemy ją jak punkt, bo jest bardzo mała w porównaniu z odległościami),
- **niebieską przerywaną linię** – „drogę po osiach": najpierw $x$ metrów wzdłuż osi $x$, potem $y$ metrów równolegle do osi $y$, na koniec $z$ metrów w górę,
- **czerwoną strzałkę** – to jest **wektor położenia** $\vec{r}$. Jego grot wskazuje **środek** pszczoły.

> ✏️ **Zadanie 1.** Zmień liczby `x`, `y`, `z` (np. na `-2`, `4`, `3`) i uruchom komórkę jeszcze raz. Co się dzieje, gdy któraś z liczb jest ujemna? A gdy wszystkie są równe `0`?

> ✏️ **Zadanie 2.** Kliknij przycisk **Z góry**. Którą współrzędną pszczoły *przestałeś widzieć*? A co zobaczysz po kliknięciu **Z przodu**? Wróć do widoku początkowego przyciskiem **↺ Reset widoku**.

> ✏️ **Zadanie 3.** Wybierz z listy **Ukryj pszczołę**. Czy sam wektor $\vec{r}$ wystarcza, żeby odczytać położenie? Potem wybierz **Ukryj wektor r** – czy sama pszczoła i niebieska linia wystarczą?

## 4. Wektor położenia $\vec{r}$

Trzy liczby $x$, $y$, $z$ opisują położenie pszczoły. Zbierzmy je w jeden obiekt – **wektor położenia**:

$$\vec{r} = (x;\ y;\ z)$$

Wektor oznaczamy **strzałką nad literą**: $\vec{r}$. To podpowiedź, że ma on nie tylko *wartość*, ale i *kierunek oraz zwrot*:

- **początek** wektora $\vec{r}$ leży w początku układu współrzędnych,
- **koniec** wektora (grot strzałki) leży w miejscu, w którym jest pszczoła,
- liczby $x$, $y$, $z$ nazywamy **składowymi** wektora.

W Pythonie wektor zapisujemy jako listę liczb w nawiasie kwadratowym, zapakowaną w `np.array(...)`:

In [ ]:
r = np.array([x, y, z])

print("Wektor położenia r =", r)
print("Składowa x:", r[0])
print("Składowa y:", r[1])
print("Składowa z:", r[2])

> 💡 **Uwaga o Pythonie:** numeracja elementów zaczyna się od **0**, nie od 1. Dlatego `r[0]` to $x$, `r[1]` to $y$, a `r[2]` to $z$.

### Jak daleko jest pszczoła od początku układu?

Długość wektora $\vec{r}$ to odległość pszczoły od początku układu. Liczymy ją z twierdzenia Pitagorasa w przestrzeni:

$$r = |\vec{r}| = \sqrt{x^2 + y^2 + z^2}$$

Zwróć uwagę: **bez strzałki** ($r$) to zwykła liczba – długość. **Ze strzałką** ($\vec{r}$) to wektor – ma kierunek i zwrot.

In [ ]:
dlugosc = np.sqrt(x**2 + y**2 + z**2)

print(f"Pszczoła jest {dlugosc:.2f} m od początku układu")

> 💡 **Uwaga o Pythonie:** `**2` oznacza „do kwadratu", `np.sqrt` to pierwiastek kwadratowy. W tekście w cudzysłowie `{...}` wstawia wartość zmiennej, a `:.2f` zaokrągla ją do dwóch miejsc po przecinku.

> ✏️ **Zadanie 4.** Policz **w głowie** długość wektora dla $\vec{r} = (3;\ 4;\ 0)$ m. Potem wpisz `x = 3`, `y = 4`, `z = 0` w komórkach wyżej i sprawdź, czy Python się z Tobą zgadza. Znasz tę trójkę liczb skądś? 😉

## 5. Bawimy się suwakami

Zamiast co chwilę zmieniać liczby w kodzie, użyjmy suwaków. Przesuwaj je i obserwuj, jak zmienia się pszczoła i wektor $\vec{r}$. Pola wyboru pod suwakami pozwalają ukryć pszczołę lub wektor. Wykres możesz przy tym obracać – widok **nie zresetuje się** po przesunięciu suwaka.

In [ ]:
pszczola_z_suwakami()

> ⚠️ Jeśli suwaki się pojawiły, ale wykres nie reaguje – uruchom komórkę ponownie (czasem Colab potrzebuje chwili na przygotowanie widżetów).

> ✏️ **Zadanie 5 (eksperymenty).** Ustaw suwaki tak, aby:
> 1. pszczoła leżała **na osi $x$** (które składowe muszą być zerowe?),
> 2. pszczoła leżała **w płaszczyźnie $xy$** (czyli „na podłodze"),
> 3. wektor $\vec{r}$ był skierowany **dokładnie w dół** od początku układu,
> 4. pszczoła była **w tej samej odległości** od początku, co na początku, ale w zupełnie innym miejscu. Ile jest takich miejsc?

> 🤔 **Pytanie do przemyślenia.** Czy położenie pszczoły zależy od tego, jak ustawimy układ współrzędnych? Co się stanie z liczbami $x$, $y$, $z$, jeśli przesuniemy początek układu o 1 m w bok? A z samą pszczołą?

## 6. Podsumowanie

| Pojęcie | Znaczenie | W Pythonie |
|---|---|---|
| Układ odniesienia | początek + trzy osie $x$, $y$, $z$ | – |
| Współrzędne | trzy liczby opisujące położenie | `x = 3` |
| Wektor położenia $\vec{r}$ | strzałka od początku układu do ciała | `r = np.array([x, y, z])` |
| Długość wektora $r = \lvert\vec{r}\rvert$ | odległość ciała od początku układu | `np.sqrt(x**2 + y**2 + z**2)` |

**W następnej części:** pszczoła zaczyna *lecieć*. Zobaczymy, jak zmienia się wektor $\vec{r}$ w czasie, i zdefiniujemy **przemieszczenie** oraz **prędkość**.

---
### 🔎 Dla ciekawskich: co robi kod pomocniczy?
Rozwiń komórkę „Kod pomocniczy" (dwukrotne kliknięcie w jej tytuł albo ikonka `<>`). Rysunek powstaje z biblioteki `plotly`: kula w paski to `go.Surface`, linie to `go.Scatter3d`, a grot strzałki to stożek zbudowany z trójkątów (`go.Mesh3d`). Spróbuj zmienić kolory pszczoły albo jej rozmiar `R_PSZCZOLY`.

---

[📚 Spis lekcji](https://github.com/TWOJ_LOGIN/NAZWA_REPO#spis-lekcji) &nbsp;·&nbsp; **Lekcja 1 z 2** &nbsp;·&nbsp; [Lekcja 2: Prędkość →](https://colab.research.google.com/github/TWOJ_LOGIN/NAZWA_REPO/blob/main/Kinematyka_2_predkosc.ipynb)